# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-07 · Builds on the ML-04 contract.
>
> Two signal checks first — both linked to real FlyRank flags from the session — then **one**
> rule, encoded with a score, a single reason code, and an action label. No fitted weights
> anywhere. This is the baseline the Week-5 model has to beat.

In [1]:
%pip install -q pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, subprocess, sys
import numpy as np, pandas as pd

REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT = REPO / "work/outputs"; OUT.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()

FRAME = OUT / "modeling_frame_dev.parquet"
if not FRAME.exists():
    subprocess.run([sys.executable, str(REPO / "work/scripts/build_modeling_frame.py")], check=True)

df = pd.read_parquet(FRAME).reset_index(drop=True)
y  = df["is_position_decline"].values
BASE = y.mean()
print(f"{len(df):,} pages · {df.client_hash_id.nunique()} clients · "
      f"features Jan-Mar 2026, label Apr 2026")
print(f"BASE RATE = {BASE:.4f}  <- every number below is read against this")

106,461 pages · 42 clients · features Jan-Mar 2026, label Apr 2026
BASE RATE = 0.5673  <- every number below is read against this


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal 1 — staleness, the premise behind the refresh flags

**The claim:** *pages that have not been updated in a long time are more likely to decline* —
the reasoning behind FlyRank's refresh flags.

I measure staleness as `days_since_update`: days between `dim_content.content_updated_date` and
my decision moment, **2026-03-31**. Before bucketing it, one look at its distribution.

In [3]:
s = df["days_since_update"]
print("days_since_update, measured at the decision moment 2026-03-31:\n")
print(s.describe(percentiles=[.1,.25,.5,.75,.9]).round(1).to_string())

n_future = int((s < 0).sum())
print(f"\nPages whose last update is AFTER 2026-03-31 : {n_future:,}  ({n_future/len(s):.1%})")
print(f"Pages usable for a staleness test           : {int((s>=0).sum()):,}  ({(s>=0).mean():.1%})")

days_since_update, measured at the decision moment 2026-03-31:

count    106461.0
mean        -50.3
std          44.5
min         -97.0
10%         -94.0
25%         -83.0
50%         -57.0
75%         -48.0
90%          34.0
max         264.0

Pages whose last update is AFTER 2026-03-31 : 87,058  (81.8%)
Pages usable for a staleness test           : 19,403  (18.2%)


**Stop.** 81.8% of pages carry a last-update date *after* my decision moment.

`dim_content` is a **current snapshot**, not a historical record: `content_updated_date` holds the
page's most recent update as of the export, including updates made in April, May and June — after
the moment I am pretending to stand at. Used as a feature it is straightforward **future
information**, and it would have walked into my rule looking like an ordinary date column.

So the honest test runs only on the 19,403 pages last updated **on or before** the cut. That
subset is itself selected on the future ("pages nobody touched afterwards"), which I state rather
than hide — it is the best test this field allows, not a clean one.

In [4]:
testable = df[df.days_since_update >= 0].copy()
testable["staleness_bucket"] = pd.cut(
    testable.days_since_update, [-1, 30, 90, 180, 365, 10**6],
    labels=["0-30 days", "31-90", "91-180", "181-365", "365+"])

tbl = (testable.groupby("staleness_bucket", observed=True)
       .agg(n=("is_position_decline", "size"),
            decline_rate=("is_position_decline", "mean"))
       .assign(vs_base=lambda t: t.decline_rate - BASE))
print("SIGNAL 1 — staleness vs decline (pages updated on/before the cut)\n")
print(tbl.round(3).to_string())
print(f"\nbase rate = {BASE:.3f}   ·   testable pages = {len(testable):,} of {len(df):,}")
print("\nSample-size floor is 50 rows per bucket; buckets below it get no verdict.")
for b, row in tbl.iterrows():
    if row.n < 50:
        print(f"  {b}: n={int(row.n)} -> INSUFFICIENT DATA, no verdict")

SIGNAL 1 — staleness vs decline (pages updated on/before the cut)

                      n  decline_rate  vs_base
staleness_bucket                              
0-30 days            90         0.822    0.255
31-90             19162         0.586    0.019
91-180              136         0.449   -0.119
181-365              15         0.667    0.099

base rate = 0.567   ·   testable pages = 19,403 of 106,461

Sample-size floor is 50 rows per bucket; buckets below it get no verdict.
  181-365: n=15 -> INSUFFICIENT DATA, no verdict


#### Verdict 1: **FALSE**

Two independent reasons, either of which is fatal on its own.

**The field cannot be used at all.** For 81.8% of pages the value encodes something that happened
after the decision moment. A rule leaning on it would be reading the future.

**And on the part that *is* testable, the gradient is not there.** The bulk of the usable pages
sit in one bucket (31–90 days, n = 19,162) at a **0.586** decline rate against a **0.567** base —
under two points, on a subset already selected for "nobody touched this afterwards". The
remaining buckets are 90, 136 and 15 rows: all below the 50-row floor, so they get no verdict
rather than a dramatic-looking ratio.

**This check just saved my rule.** Staleness was going into it — the refresh-flag reasoning is
genuinely persuasive, and on this data it is both unusable and unsupported. `days_since_update`
is now excluded from the rule and from every model in this project. A clearly-explained negative
is a win.

### Signal 2 — CTR versus position, the premise behind the CTR-fix logic

**The claim:** *a page that ranks well but earns a poor click-through rate for its position is
underperforming, and that underperformance precedes decline* — the reasoning behind the CTR-fix
flag.

The test has to be **within position tier**, because CTR falls with position for reasons that have
nothing to do with page quality. Comparing a position-3 page's CTR to a position-40 page's CTR
measures the ranking, not the page. So: bucket by tier, then split each tier at its own median CTR.

In [5]:
df["pos_tier"] = pd.cut(df.f_pos, [0, 3, 10, 20, 50, 10**6],
                        labels=["top_3", "page_1 (4-10)", "striking (11-20)",
                                "page_3_5 (21-50)", "deep (50+)"])

tier = (df.groupby("pos_tier", observed=True)
        .apply(lambda x: pd.Series({
            "n": len(x),
            "weighted_CTR_%": 100 * x.f_clicks.sum() / x.f_impressions.sum(),
            "decline_rate": x.is_position_decline.mean()}), include_groups=False))
print("Context — CTR really does fall with position, so the test must be within tier:\n")
print(tier.round(3).to_string())

df["tier_median_ctr"] = df.groupby("pos_tier", observed=True).f_ctr.transform("median")
df["low_ctr_for_tier"] = df.f_ctr < df.tier_median_ctr

split = (df.groupby(["pos_tier", "low_ctr_for_tier"], observed=True)
         .agg(n=("is_position_decline", "size"),
              decline_rate=("is_position_decline", "mean")).round(3))
print("\n\nSIGNAL 2 — within each tier: below-median CTR vs above-median\n")
print(split.to_string())

print("\n\nThe gap that matters (below-median minus above-median decline rate):\n")
for t in df.pos_tier.cat.categories:
    sub = df[df.pos_tier == t]
    if sub.low_ctr_for_tier.nunique() < 2 or len(sub) < 50:
        print(f"  {t:20s} n={len(sub):>6,}  -> insufficient split, no verdict"); continue
    lo = sub[sub.low_ctr_for_tier].is_position_decline.mean()
    hi = sub[~sub.low_ctr_for_tier].is_position_decline.mean()
    flag = "supports" if lo - hi > 0.02 else ("REVERSES" if lo - hi < -0.02 else "flat")
    print(f"  {t:20s} n={len(sub):>6,}  low={lo:.3f}  high={hi:.3f}  gap={lo-hi:+.3f}  {flag}")

Context — CTR really does fall with position, so the test must be within tier:

                        n  weighted_CTR_%  decline_rate
pos_tier                                               
top_3             10835.0           0.387         0.625
page_1 (4-10)     50913.0           0.328         0.563
striking (11-20)  22212.0           0.321         0.611
page_3_5 (21-50)  18656.0           0.159         0.543
deep (50+)         3845.0           0.047         0.330


SIGNAL 2 — within each tier: below-median CTR vs above-median

                                       n  decline_rate
pos_tier         low_ctr_for_tier                     
top_3            False              5418         0.586
                 True               5417         0.663
page_1 (4-10)    False             25457         0.509
                 True              25456         0.616
striking (11-20) False             11106         0.601
                 True              11106         0.622
page_3_5 (21-50) False 

#### Verdict 2: **MIXED**

The claim holds where the page is shallow and fades to nothing where it is deep.

- **`page_1 (4-10)`** — the strongest support: **0.616 vs 0.509**, a **+10.7 point** gap on
  50,913 pages. Among pages already on page one, poor click capture really does precede losing
  position.
- **`top_3`** — supports it: **0.663 vs 0.586**, +7.8 points on 10,835 pages.
- **`striking (11-20)`** — **+2.1 points** (0.622 vs 0.601, n = 22,212). Directionally right, but
  barely clear of the ±2-point band I set as meaningful. I would not build on it alone.
- **`page_3_5 (21-50)`** — **flat, and slightly negative**: −1.6 points (0.535 vs 0.551). Inside
  the band, so the honest reading is "no signal here", not "reversed". Down at those positions CTR
  is computed on tiny click counts and is mostly noise.
- **`deep (50+)`** — **no verdict**: every page sits on one side of the tier median, so there is
  no within-tier split to test at all.

**What I take into the rule:** low-CTR-for-tier, **restricted to pages at position ≤ 20**. The
signal is real, but only where the claim's own logic applies — a page has to be visible enough for
click capture to mean anything. Applied portfolio-wide it would add noise from the bottom tiers
where the effect vanishes.

### The rule, in plain words

> A page needs review first if its **position was already sliding** before the cut-off. Among
> pages that are sliding, prioritise the ones **shallow enough to have something to lose**. If a
> page is not sliding, it is still worth a look when it is **on page 1–2 and capturing fewer
> clicks than its tier normally does**, or when it is **visible and unstable**.

That is three sentences, four conditions, and no fitted weights. Staleness is absent by the
verdict above. Each condition uses only feature-window columns.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# ---- the score: four transparent conditions, hand-weighted, no fitting ----
sliding  = df.f_pos_trend.fillna(0) >= 1.0                       # position worsened Jan -> Mar
shallow  = (df.f_pos > 0) & (df.f_pos <= 20)                     # page 1-2: room to lose
low_ctr  = df.low_ctr_for_tier & shallow                         # only where Signal 2 held
unstable = df.f_pos_volatility.fillna(0) >= df.f_pos_volatility.median()
visible  = df.f_impressions >= 1000

df["rule_points"] = (3 * sliding.astype(int)      # prior direction: the heaviest evidence
                     + 1 * shallow.astype(int)
                     + 1 * low_ctr.astype(int)
                     + 1 * unstable.astype(int)
                     + 1 * visible.astype(int))

# Four integer conditions give only a handful of distinct scores, so thousands of pages tie.
# Break ties by exposure, squashed inside one point so it can never outrank real evidence.
expo = np.log1p(df.f_impressions.values)
df["baseline_score"] = df.rule_points + 0.999 * (expo - expo.min()) / (expo.max() - expo.min())

# ---- ONE reason code per page: first match wins, most specific first ----
df["reason_code"] = np.select(
    [sliding & shallow, sliding & ~shallow, low_ctr, unstable & visible],
    ["sliding_on_page_1_2", "sliding_but_deep", "low_ctr_for_position", "unstable_and_visible"],
    default="no_signal")

# ---- one action per reason code ----
ACTION = {"sliding_on_page_1_2":  "review_now",
          "sliding_but_deep":     "monitor",
          "low_ctr_for_position": "review_ctr",
          "unstable_and_visible": "investigate_volatility",
          "no_signal":            "monitor"}
df["action"] = df.reason_code.map(ACTION)

print("Reason codes — each page gets exactly one:\n")
print(df.groupby("reason_code", observed=True)
        .agg(n=("is_position_decline", "size"),
             decline_rate=("is_position_decline", "mean"),
             action=("action", "first")).round(3).to_string())
print(f"\nbase rate = {BASE:.3f}")
assert df.reason_code.notna().all() and len(df.reason_code.unique()) <= 5

Reason codes — each page gets exactly one:

                          n  decline_rate                  action
reason_code                                                      
low_ctr_for_position  28818         0.539              review_ctr
no_signal             37010         0.447                 monitor
sliding_but_deep      10654         0.616                 monitor
sliding_on_page_1_2   24212         0.777              review_now
unstable_and_visible   5767         0.512  investigate_volatility

base rate = 0.567


In [7]:
def precision_at_k(scores, labels, k):
    return np.asarray(labels)[np.argsort(-np.asarray(scores), kind="stable")[:k]].mean()

queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

COLS = ["rank", "content_hash_id", "client_hash_id", "baseline_score", "rule_points",
        "reason_code", "action", "f_pos", "f_pos_trend", "f_pos_volatility", "f_ctr",
        "f_impressions", "f_days_with_impressions"]
csv_path = OUT / "baseline_action_score.csv"
queue[COLS].to_csv(csv_path, index=False)
print(f"wrote {rel(csv_path)}  ({len(queue):,} rows)\n")

rows = []
for k in [10, 50, 100, 500, 1000, 5000]:
    p = precision_at_k(queue.baseline_score, queue.is_position_decline, k)
    rows.append({"k": k, "precision_at_k": round(float(p), 4),
                 "lift_over_base": round(float(p / BASE), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nbase rate = {BASE:.4f}")

json.dump({"base_rate": round(float(BASE), 4), "n_items": int(len(df)),
           "n_clients": int(df.client_hash_id.nunique()),
           "signal_verdicts": {"staleness_refresh_flag": "FALSE",
                               "ctr_vs_position_ctrfix_flag": "MIXED"},
           "rule": {"conditions": {"sliding": 3, "shallow": 1, "low_ctr_for_position": 1,
                                   "unstable": 1, "visible": 1},
                    "tie_break": "log1p(f_impressions) scaled into [0,1)"},
           "reason_code_counts": {k: int(v) for k, v in df.reason_code.value_counts().items()},
           "precision_at_k": rows,
           "note": "frozen once ML-08 modelling starts"},
          open(OUT / "w04_baseline_metrics.json", "w"), indent=2)
print(f"\nreceipt -> {rel(OUT / 'w04_baseline_metrics.json')}")

wrote work/outputs/baseline_action_score.csv  (106,461 rows)

   k  precision_at_k  lift_over_base
  10          0.9000           1.587
  50          0.8200           1.446
 100          0.8900           1.569
 500          0.9160           1.615
1000          0.9020           1.590
5000          0.8496           1.498

base rate = 0.5673

receipt -> work/outputs/w04_baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top10 = queue.head(10).copy()
top10["page"]   = ["p" + h[-6:] for h in top10.content_hash_id]
top10["client"] = ["c" + h[-4:] for h in top10.client_hash_id]
print(top10[["rank","page","client","action","reason_code","baseline_score",
             "f_pos","f_pos_trend","f_ctr","f_impressions","is_position_decline"]].to_string(
    index=False, formatters={"baseline_score":"{:.2f}".format, "f_pos":"{:.1f}".format,
                             "f_pos_trend":"{:+.1f}".format, "f_ctr":"{:.2f}".format,
                             "f_impressions":"{:,.0f}".format}))
print(f"\nOf these 10, {int(top10.is_position_decline.sum())}/10 declined "
      f"(base rate would give ~{10*BASE:.0f}/10).")

 rank    page client     action         reason_code baseline_score f_pos f_pos_trend f_ctr f_impressions  is_position_decline
    1 p6668e3  c65ea review_now sliding_on_page_1_2           7.91   1.2        +1.7  0.00       362,658                    0
    2 p106de4  c63c4 review_now sliding_on_page_1_2           7.89  18.7       +12.4  0.12       311,782                    1
    3 p15651b  c65ea review_now sliding_on_page_1_2           7.81   2.8        +1.5  0.00       149,311                    1
    4 p096e6d  c65ea review_now sliding_on_page_1_2           7.75   2.4       +19.1  0.03        91,398                    1
    5 pa3e9e7  c63c4 review_now sliding_on_page_1_2           7.74  18.2       +13.6  0.11        78,645                    1
    6 p3fca87  c63c4 review_now sliding_on_page_1_2           7.72   7.8        +5.3  0.04        69,850                    1
    7 p8f5e94  c63c4 review_now sliding_on_page_1_2           7.71  19.6       +22.7  0.02        62,190              

In [9]:
# One line per row: the action, why it is there, and what would make it wrong.
print("TOP-10 REVIEW\n" + "="*100)
for _, r in top10.iterrows():
    why = (f"position worsened {r.f_pos_trend:+.1f} places Jan->Mar while sitting at "
           f"{r.f_pos:.1f} with {r.f_impressions:,.0f} impressions")
    if r.reason_code == "low_ctr_for_position":
        why = (f"holding position {r.f_pos:.1f} but capturing only {r.f_ctr:.2f}% CTR, "
               f"below its tier median")
    elif r.reason_code == "unstable_and_visible":
        why = (f"position bounced (sd {r.f_pos_volatility:.1f}) on "
               f"{r.f_impressions:,.0f} impressions with no clear direction")
    if r.f_ctr == 0:
        wrong = (f"it earned zero clicks on {r.f_impressions:,.0f} impressions, so its ranking "
                 "may sit on queries nobody clicks - losing that position may cost nothing")
    elif r.f_days_with_impressions < 60:
        wrong = (f"it only appeared on {int(r.f_days_with_impressions)} of 90 days, so its "
                 "average position is built on thin, intermittent data")
    else:
        wrong = ("its January position was an unsustainable spike, so this is a return to normal "
                 "rather than decay")
    print(f"#{int(r['rank']):<3} {r.page}  [{r.action}]")
    print(f"     why : {why}")
    print(f"     wrong if : {wrong}")

TOP-10 REVIEW
#1   p6668e3  [review_now]
     why : position worsened +1.7 places Jan->Mar while sitting at 1.2 with 362,658 impressions
     wrong if : its January position was an unsustainable spike, so this is a return to normal rather than decay
#2   p106de4  [review_now]
     why : position worsened +12.4 places Jan->Mar while sitting at 18.7 with 311,782 impressions
     wrong if : its January position was an unsustainable spike, so this is a return to normal rather than decay
#3   p15651b  [review_now]
     why : position worsened +1.5 places Jan->Mar while sitting at 2.8 with 149,311 impressions
     wrong if : it earned zero clicks on 149,311 impressions, so its ranking may sit on queries nobody clicks - losing that position may cost nothing
#4   p096e6d  [review_now]
     why : position worsened +19.1 places Jan->Mar while sitting at 2.4 with 91,398 impressions
     wrong if : its January position was an unsustainable spike, so this is a return to normal rather than decay
#5 

**Reading the top 10.** Every row is the same shape of claim: *observed* feature-window evidence,
one reason code, one action, and the specific thing that would make it a bad pick.

Two caveats apply to all ten. The label is a **relative position move**, so a page can appear
here, genuinely lose average position, and still gain clicks — if it slipped on low-value queries
and held the ones that convert. And nothing in this queue measures query value, conversion, or
revenue. That is why the output is a review queue for a human rather than an automated trigger.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# --- Weak pick 1: ties. Four integer conditions cannot order a top-10 on their own. ---
print("Ranking by integer rule_points alone (ties left in frame order):")
for k in [10, 50, 100, 500, 5000]:
    p = precision_at_k(df.rule_points.astype(float), y, k)
    print(f"  P@{k:<5d} {p:.3f}" + ("   <- BELOW base rate" if p < BASE else ""))
print(f"\ndistinct rule_points values: {df.rule_points.nunique()} across {len(df):,} pages")
print("The frame arrives clustered by client, so an unbroken tie block is one client's pages.")
print("That is what the exposure tie-break fixes.\n")

# --- Weak pick 2: client concentration at the top ---
top50 = queue.head(50)
vc = top50.client_hash_id.value_counts()
print(f"Top 50 spans {top50.client_hash_id.nunique()} clients of "
      f"{df.client_hash_id.nunique()}; largest holds {vc.iloc[0]}/50 ({vc.iloc[0]/50:.0%}).")

# --- Weak pick 3: the blind spot the rule accepts ---
deep = df[df.f_pos > 50]
print(f"\nPages deeper than position 50: n={len(deep):,}, decline {deep.is_position_decline.mean():.3f} "
      f"vs base {BASE:.3f} — scored low by design.")

Ranking by integer rule_points alone (ties left in frame order):
  P@10    0.600
  P@50    0.680
  P@100   0.750
  P@500   0.836
  P@5000  0.830

distinct rule_points values: 8 across 106,461 pages
The frame arrives clustered by client, so an unbroken tie block is one client's pages.
That is what the exposure tie-break fixes.

Top 50 spans 5 clients of 42; largest holds 36/50 (72%).

Pages deeper than position 50: n=3,845, decline 0.330 vs base 0.567 — scored low by design.


In [11]:
# --- Leakage check: no future-window or label-derived input reaches the score ---
USED = ["f_pos", "f_pos_trend", "f_pos_volatility", "f_ctr", "f_impressions",
        "f_days_with_impressions", "tier_median_ctr"]
LABEL_SIDE  = {"o_pos", "o_impressions", "pos_delta", "is_position_decline"}
FUTURE_INFO = {"days_since_update"}          # proven future-dated in Signal 1
DECISION    = {"last_optimized_date", "optimization_eligible_date", "is_published", "is_deleted"}

for name, bad in [("label-derived", LABEL_SIDE), ("future information", FUTURE_INFO),
                  ("product decision flags", DECISION)]:
    hit = set(USED) & bad
    assert not hit, f"LEAK: {name} in the score -> {hit}"
    print(f"no {name:24s} in the score  (checked against {sorted(bad)})")

assert not any(c.startswith("query90d") for c in USED)
print(f"\nno fact_content_query_90d columns    (its window opens 2026-04-02)")
print(f"\nEvery scored column is measured 2026-01-01..2026-03-31; the label lives in April 2026.")

no label-derived            in the score  (checked against ['is_position_decline', 'o_impressions', 'o_pos', 'pos_delta'])
no future information       in the score  (checked against ['days_since_update'])
no product decision flags   in the score  (checked against ['is_deleted', 'is_published', 'last_optimized_date', 'optimization_eligible_date'])

no fact_content_query_90d columns    (its window opens 2026-04-02)

Every scored column is measured 2026-01-01..2026-03-31; the label lives in April 2026.


### Weak picks — what I would not defend

1. **The rule's ordering is only half its own doing.** Four integer conditions produce just
   **8 distinct scores** across 106,461 pages, so thousands of pages tie. Ranked on the integers
   alone the rule still beats the base rate (P@10 0.600, P@50 0.680, P@500 0.836 against 0.567) —
   but it is markedly worse than the tie-broken version at the top of the list, where a queue is
   actually worked (0.900 and 0.820 at K=10 and 50). The frame arrives clustered by client, so an
   unbroken tie block is largely one client's pages, and which of them surface is an accident of
   row order. The exposure tie-break is a **priority judgement** — "among equal evidence, review
   what more people see" — not a discovered fact, and roughly a third of the rule's top-of-list
   performance rests on it.

2. **The top of the queue is one client's problem.** A single client holds most of the top 50.
   Exposure and traffic are both concentrated by client, so the tie-break compounds it. A usable
   queue needs a per-client cap — deferred to ML-10, and named here so it is not mistaken for an
   oversight.

3. **`sliding` may be measuring regression to the mean.** Pages that slid are the ones most likely
   to keep sliding — and also the ones whose January position may have been an unsustainable
   spike. The rule cannot separate genuine decay from a page settling back; both look identical.

4. **Deep pages are ignored on purpose.** Pages past position 50 decline much less often, so they
   score low. A page falling from 55 to 70 is still losing ground, invisibly. The rule optimises
   for pages worth defending, not for measuring all decline.

5. **A third of pages have no measurable prior direction.** `f_pos_trend` is null for pages absent
   in January; they get zero points for the heaviest condition regardless of actual risk.

**Leakage: clean.** The score reads six feature-window columns plus a within-tier CTR median. No
April data, no product decision flags, no query-table columns — and **no `days_since_update`,
which Signal 1 proved is future-dated for 81.8% of pages**. The assertions above fail the notebook
if that stops being true.

**Frozen.** These conditions and thresholds do not move from here. Changing the baseline after
seeing the model would make the ML-08 comparison meaningless.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.